In [1]:
%pip install requests pandas

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached pandas-2.3.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached charset_normalizer-3.4.3-cp311-cp311-macosx_10_9_universal2.whl.metadata (36 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached certifi-2025.8.3-py3-none-any.whl.metadata (2.4 kB)
  Using cached numpy-2.3.2-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.3-cp311-cp311-macosx_10_9_universal2.whl (204 kB)
Using cached idna-3.10-py3-none-any.whl (70 kB)
Using cached urllib3-2.5.0-py3-none-any.whl (129 kB)
Using cached pandas-2.3.2-cp311-cp311-macosx_11_0_arm64.whl (10.8 MB)
Using cached certifi-2025.8.3-py3-none-any.whl 

In [2]:
import requests
from collections import defaultdict
import json
import time
import pandas as pd
from constants import GITHUB_ORG, GITHUB_TOKEN

In [12]:
BASE_URL = 'https://api.github.com'

headers = {
    'Authorization' : f'token {GITHUB_TOKEN}',
    'Accept' : 'application/vnd.github.v3+json'
}

In [15]:
def get_repositories(org_name):
    url = f"{BASE_URL}/orgs/{org_name}/repos"
    response = requests.get(url, headers=headers)
    # print("get_repositories: " + str(response.status_code))
    return response.json() if response.status_code == 200 else []

def get_branches(repo_name):
    url = f"{BASE_URL}/repos/{GITHUB_ORG}/{repo_name}/branches"
    response = requests.get(url, headers=headers)
    print(response.json() if response.status_code == 200 else response.status_code)
    # print("get_branches: " + str(response.status_code))
    return response.json() if response.status_code == 200 else []

def get_commits(repo_name, branch_name):
    url = f"{BASE_URL}/repos/{GITHUB_ORG}/{repo_name}/commits?sha={branch_name}"
    response = requests.get(url, headers=headers)
    # print("get_commits: " + str(response.status_code))
    return response.json() if response.status_code == 200 else []

def get_commit_stats(repo_name, commit_sha):
    url = f"{BASE_URL}/repos/{GITHUB_ORG}/{repo_name}/commits/{commit_sha}"
    response = requests.get(url, headers=headers)
    # print("get_commit_stats: " + str(response.status_code))
    if response.status_code == 200:
        data = response.json()
        file_data = data.get('files', '')
        files = []
        additions = 0
        deletions = 0
        for file in file_data:
            files.append(file['filename'])
            additions += file['additions']
            deletions += file['deletions']
        return {
            'files' : files,
            'additions' : additions,
            'deletions' : deletions,
        }
    return {'additions': 0, 'deletions': 0}

def collect_commit_data():
    commit_data_by_user = defaultdict(list)
    teams = pd.read_csv('team_mapping.csv')
    teams['teamID'].tolist()
    repositories = teams['teamID'].tolist()
    k = 0
    for repo in repositories:
        print("running ", repo)
        branches = get_branches(repo)
        for branch in branches:
            branch_name = branch['name']
            print("on branch ", branch_name)
            commits = get_commits(repo, branch_name)

            for commit in commits:
                author = commit.get('commit', {}).get('author', {}).get('email', 'Unknown')
                message = commit.get('commit', {}).get('message', '')
                timestamp = commit.get('commit', {}).get('author', {}).get('date', '')
                commit_sha = commit.get('sha', '')
                url = commit.get('url', '')
                diff_url = commit.get('commit', {}).get('diff_url', '')
                # Fetch additions and deletions using the commit SHA
                stats = get_commit_stats(repo, commit_sha)
                file_names = []
                if "files" in stats.keys():
                    file_names = stats['files']
                commit_data_by_user[repo].append({
                    'login' : author,
                    'sha' : commit_sha,
                    'url' : url,
                    'timestamp' : timestamp,
                    'branch': branch_name,
                    'message': message,
                    'files' : file_names,
                    'additions': stats['additions'],
                    'deletions': stats['deletions'],
                    'diff_url': diff_url
                })
        k += 1
        print('Team #' + str(k) + ': ' + repo + ' data aggregated successfully.')
        # if k % 5 == 0:
        time.sleep(5)
    return commit_data_by_user

In [16]:
commit_data_by_user = collect_commit_data()
print("Commit history has been collected and saved to commit_history.json")
# Save data to JSON file
with open("commit_history.json", "w") as file:
    json.dump(commit_data_by_user, file, indent=4)

running  AD1_1
403
Team #1: AD1_1 data aggregated successfully.
running  AD1_2
403
Team #2: AD1_2 data aggregated successfully.
Commit history has been collected and saved to commit_history.json
